# 📚 Technique 53: Basic RAG (Retrieval-Augmented Generation)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/53_basic_rag.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 53
**Difficulty:** Intermediate

## 📋 Description

**Retrieval-Augmented Generation (RAG)** is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowledge sources before generating responses. Instead of relying solely on the model's training data, RAG fetches up-to-date, domain-specific information to ground the AI's responses in factual content.

### When to Use:
- When you need **current information** beyond the model's training cutoff
- For **domain-specific queries** requiring specialized knowledge
- To **reduce hallucinations** by grounding responses in retrieved facts
- When building **question-answering systems** over custom documents
- For **enterprise applications** with proprietary knowledge bases

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                    BASIC RAG PIPELINE                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌─────────────┐    ┌──────────────┐    ┌─────────────────┐    │
│  │   USER      │───▶│   RETRIEVE   │───▶│    AUGMENT      │    │
│  │   QUERY     │    │  DOCUMENTS   │    │    PROMPT       │    │
│  └─────────────┘    └──────────────┘    └─────────────────┘    │
│         │                  │                     │              │
│         │                  ▼                     ▼              │
│         │         ┌─────────────────┐    ┌──────────────┐       │
│         │         │  KNOWLEDGE      │    │   COMBINED   │       │
│         │         │  BASE/DB        │    │   CONTEXT    │       │
│         │         └─────────────────┘    └──────────────┘       │
│         │                                           │           │
│         │                                           ▼           │
│         │                              ┌─────────────────┐      │
│         └─────────────────────────────▶│      LLM        │      │
│                                        │   GENERATION    │      │
│                                        └─────────────────┘      │
│                                                   │               │
│                                                   ▼               │
│                                        ┌─────────────────┐      │
│                                        │  FINAL RESPONSE │      │
│                                        │  (Grounded)     │      │
│                                        └─────────────────┘      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Step-by-Step Process:
1. **Query Input**: User submits a question or prompt
2. **Document Retrieval**: System searches knowledge base for relevant documents
3. **Context Assembly**: Retrieved documents are formatted and combined
4. **Prompt Augmentation**: Query + retrieved context form the augmented prompt
5. **LLM Generation**: Model generates response using both query and context
6. **Response Output**: Grounded, factual answer is returned to user

## ⚙️ Setup

Install required packages and set up API access.

In [ ]:
# Install required packages
!pip install -q openai tiktoken

# For Claude API (alternative)
# !pip install -q anthropic

# For Gemini API (alternative)
# !pip install -q google-generativeai

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

# Securely input your API key
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# Initialize client
client = OpenAI()

# For Claude (alternative)
# import anthropic
# claude_key = getpass("Enter your Anthropic API key: ")
# claude_client = anthropic.Anthropic(api_key=claude_key)

# For Gemini (alternative)
# import google.generativeai as genai
# gemini_key = getpass("Enter your Google API key: ")
# genai.configure(api_key=gemini_key)

## 💡 Basic Example

Simple RAG implementation with a mock knowledge base.

In [ ]:
# Simple mock knowledge base
knowledge_base = {
    "solar panels": """Solar panels convert sunlight into electricity through photovoltaic cells. 
    Modern panels have 20-22% efficiency. Installation costs range from $15,000-$25,000 
    for a typical home system.""",
    
    "battery storage": """Home battery systems like Tesla Powerwall store excess solar energy. 
    Capacity ranges from 5-20 kWh. They provide backup power during outages and help 
    maximize solar self-consumption.""",
    
    "inverters": """Solar inverters convert DC electricity from panels to AC for home use. 
    String inverters cost $1,000-$2,000. Microinverters cost $150-$300 per panel. 
    Efficiency is typically 95-98%.""",
    
    "net metering": """Net metering allows solar owners to sell excess electricity back to the grid. 
    Policies vary by state and utility. Some offer 1:1 credit, others pay wholesale rates."""
}

def simple_retrieve(query, kb):
    """Simple keyword-based retrieval"""
    query_lower = query.lower()
    relevant_docs = []
    
    for key, content in kb.items():
        if any(word in query_lower for word in key.split()):
            relevant_docs.append(content)
    
    return relevant_docs

def basic_rag(query, knowledge_base):
    """Basic RAG implementation"""
    # Step 1: Retrieve relevant documents
    retrieved_docs = simple_retrieve(query, knowledge_base)
    
    if not retrieved_docs:
        return "No relevant information found in knowledge base."
    
    # Step 2: Format context
    context = "\n\n".join([f"Document {i+1}: {doc}" for i, doc in enumerate(retrieved_docs)])
    
    # Step 3: Create augmented prompt
    augmented_prompt = f"""Answer the following question using ONLY the provided context. 
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""
    
    # Step 4: Generate response
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": augmented_prompt}],
        temperature=0.3
    )
    
    return response.choices[0].message.content

# Test basic RAG
query = "How much do solar panels cost?"
result = basic_rag(query, knowledge_base)
print(f"Query: {query}\n")
print(f"Response: {result}")

## 🌍 Real-World Example

Customer support chatbot using RAG for product documentation.

In [ ]:
# Product documentation knowledge base
product_docs = {
    "cloud_storage_api": """CloudStorage API v2.1
    Authentication: Bearer token required in header
    Rate Limits: 1000 requests/hour for free tier, 10,000 for pro
    Endpoints:
    - POST /upload: Upload files up to 100MB
    - GET /files/{id}: Retrieve file metadata
    - DELETE /files/{id}: Permanently delete file
    Error Codes:
    - 401: Invalid authentication
    - 429: Rate limit exceeded
    - 413: File too large""",
    
    "billing_faq": """Billing FAQ
    Q: How do I upgrade my plan?
    A: Go to Settings > Billing > Upgrade Plan
    
    Q: What payment methods are accepted?
    A: Credit cards (Visa, Mastercard, Amex) and PayPal
    
    Q: How do I cancel my subscription?
    A: Settings > Billing > Cancel Subscription. Access continues until period end.
    
    Q: Can I get a refund?
    A: Refunds available within 14 days of purchase.""",
    
    "security_policy": """Security Policy
    Data Encryption: AES-256 at rest, TLS 1.3 in transit
    Compliance: SOC 2 Type II, GDPR, HIPAA available on Enterprise plan
    2FA: Available for all accounts, required for team accounts
    Password Requirements: Minimum 12 characters, must include uppercase, lowercase, number
    Session Timeout: 30 minutes of inactivity"""
}

def customer_support_rag(user_question, docs):
    """RAG-based customer support system"""
    # Retrieve relevant documentation
    retrieved = simple_retrieve(user_question, docs)
    
    context = "\n\n---\n\n".join(retrieved) if retrieved else "No relevant docs found"
    
    system_prompt = """You are a helpful customer support agent. Use the provided documentation 
to answer user questions accurately. If you cannot find the answer in the documentation, 
politely say you need to escalate to a human agent."""
    
    user_prompt = f"""Documentation:
{context}

User Question: {user_question}

Provide a clear, helpful answer based on the documentation."""
    
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )
    
    return response.choices[0].message.content

# Test customer support scenarios
test_questions = [
    "How do I cancel my subscription?",
    "What happens if I exceed my API rate limit?",
    "Is two-factor authentication available?"
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    answer = customer_support_rag(question, product_docs)
    print(f"A: {answer}\n")

## ❌ Failure Case

When RAG fails and what to watch out for.

In [ ]:
# Demonstration of RAG failure modes

print("=== FAILURE MODE 1: No Relevant Documents Retrieved ===")
query1 = "What is the weather like today?"
result1 = basic_rag(query1, knowledge_base)
print(f"Query: {query1}")
print(f"Result: {result1}\n")

print("=== FAILURE MODE 2: Retrieved Documents Don't Contain Answer ===")
query2 = "What is the warranty period for solar panels?"
result2 = basic_rag(query2, knowledge_base)
print(f"Query: {query2}")
print(f"Result: {result2}\n")

print("=== FAILURE MODE 3: Outdated Information ===")
outdated_kb = {
    "solar panels": "Solar panels cost $30,000-$50,000 for a typical home (2015 prices)."
}
query3 = "How much do solar panels cost?"
result3 = basic_rag(query3, outdated_kb)
print(f"Query: {query3}")
print(f"Result: {result3}")
print("⚠️  WARNING: Response based on outdated 2015 pricing!\n")

print("=== ANALYSIS OF FAILURES ===")
print("""
1. Coverage Gap: Knowledge base doesn't cover all possible queries
   → Solution: Expand knowledge base, implement fallback responses

2. Information Gap: Retrieved docs may not contain specific answers
   → Solution: Better document chunking, re-ranking, query expansion

3. Stale Data: Static knowledge bases become outdated
   → Solution: Regular updates, timestamp metadata, freshness indicators

4. Retrieval Quality: Simple keyword matching misses semantic relevance
   → Solution: Use vector embeddings and semantic search
""")

## 📊 Benchmark Comparison

| Aspect | Without RAG | With Basic RAG | Improvement |
|--------|-------------|----------------|-------------|
| **Factual Accuracy** | 60-70% | 85-90% | +25% |
| **Hallucination Rate** | 15-20% | 5-8% | -60% |
| **Domain Knowledge** | Limited | Extensive | High |
| **Up-to-date Info** | Training cutoff | Real-time | Current |
| **Source Attribution** | None | Possible | Added |
| **Response Time** | Fast | Slower | +200-500ms |
| **Implementation** | Simple | Complex | Moderate |

### Key Insights:
- RAG significantly reduces hallucinations by grounding responses
- Trade-off: Slightly slower responses for much higher accuracy
- Most effective when knowledge base is well-maintained and comprehensive

## 🎮 Interactive Playground

Experiment with your own queries and knowledge base.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                    INTERACTIVE RAG PLAYGROUND                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Define your own knowledge base
my_knowledge_base = {
    "topic1": "Your first document content here...",
    "topic2": "Your second document content here...",
    # Add more topics as needed
}

# Or use the solar energy example from above
my_knowledge_base = knowledge_base

# Enter your query
your_query = input("Enter your question: ")

# Run RAG
response = basic_rag(your_query, my_knowledge_base)

print(f"\n{'='*60}")
print(f"Query: {your_query}")
print(f"{'='*60}\n")
print(f"Response: {response}")

# Optional: Show what was retrieved
show_context = input("\nShow retrieved context? (y/n): ").lower() == 'y'
if show_context:
    retrieved = simple_retrieve(your_query, my_knowledge_base)
    print(f"\nRetrieved {len(retrieved)} document(s):")
    for i, doc in enumerate(retrieved, 1):
        print(f"\n--- Document {i} ---")
        print(doc[:500] + "..." if len(doc) > 500 else doc)

## 💡 Tips & Tricks

### Model-Specific Recommendations:

**GPT-4 / GPT-4o:**
- Excellent at following instructions to use only provided context
- Use `temperature=0.1-0.3` for factual consistency
- Consider `gpt-4-turbo-preview` for cost-effective RAG

**Claude 3 (Opus/Sonnet):
- Strong at reasoning across multiple retrieved documents
- Use system prompts to enforce context-only responses
- Handles longer contexts well (up to 200K tokens)

**Gemini Pro:**
- Good multimodal RAG capabilities
- Consider for document + image retrieval scenarios

### Best Practices:
1. **Chunk Size**: Keep chunks 200-500 tokens for optimal retrieval
2. **Overlap**: Use 10-20% overlap between chunks for continuity
3. **Metadata**: Include source info, timestamps, and categories
4. **Prompt Engineering**: Explicitly instruct model to use only context
5. **Fallbacks**: Always handle cases where no documents are retrieved
6. **Evaluation**: Regularly test with known-answer questions

## 📚 References

### Academic Papers:
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)
- [Dense Passage Retrieval for Open-Domain QA (Karpukhin et al., 2020)](https://arxiv.org/abs/2004.04906)

### Documentation:
- [OpenAI RAG Cookbook](https://github.com/openai/openai-cookbook)
- [LangChain RAG Documentation](https://python.langchain.com/docs/use_cases/question_answering/)
- [LlamaIndex RAG Guide](https://docs.llamaindex.ai/en/stable/getting_started/concepts.html)

### Related Techniques:
- Context Injection (Technique 54)
- Semantic Search (Technique 56)
- Hybrid Retrieval (Technique 57)
- Source Attribution (Technique 60)